In [15]:
import pandas as pd

In [39]:
res = pd.read_csv("audmind_test__alltools_qwen_results_only_answer.csv")
res.head(1)

,file_id,text,Question,Answer,Reasoning,hints,selected_tools,selected tools_list,selected num_tools,all toolvalues,selected toolvalues,predicted_answer,predicted_reasoning,processing_status,raw_response
0,/data/amey_2311cs10/debayan/test_mentalhealth_...,You're not sure? Do you feel like you're at r...,What cause of depression does this show?,This patient shows causes of depression relate...,The text raises a question about feeling at ri...,Pitch Variability: 94.73721; Speech Rate: 1.45...,Pitch Variability; Speech Rate; bertemotion_cl...,"['Pitch Variability', 'Speech Rate', 'bertemot...",5,"{'Pitch Variability': '94.73721', 'Speech Rate...","{'Pitch Variability': '94.73721', 'Speech Rate...",Based on the transcription and the tool analys...,No reasoning provided,success,Based on the transcription and the tool analys...


### COMET scores

In [40]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

Fetching 5 files: 100%|████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 29873.96it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.1.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/data/amey_2311cs10/anaconda3/envs/condapy312/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


#### Answer part

In [25]:
# Prepare data in COMET format
comet_reason_data = [
    {
        "src": "",  # Source left empty
        "mt": res.loc[i, "predicted_answer"],
        "ref": res.loc[i, "Answer"]
    }
    for i in range(len(res))
]

model_output = model.predict(comet_reason_data, batch_size=8, gpus=1)  # Set gpus=0 if no GPU
# Individual scores
comet_scores = model_output.scores
# Average score
avg_comet = sum(comet_scores) / len(comet_scores)

print(f"Average COMET score for ANSWER: {avg_comet:.4f}")

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Predicting DataLoader 0: 100%|████

Average COMET score for ANSWER: 0.6587


#### Reason part

### (Ans + Reasoning) actual v Pred_Ans

In [41]:
res['ans_reason'] = res['Answer'].str.cat(res['Reasoning'], sep=' ')
#res['ans_reason'] = res['Answer'] + " " + res['Reasoning']

# Prepare data in COMET format
comet_reason_data = [
    {
        "src": "",  # Source left empty
        "mt": res.loc[i, "predicted_answer"],
        "ref": res.loc[i, "ans_reason"]
    }
    for i in range(len(res))
]

model_output = model.predict(comet_reason_data, batch_size=8, gpus=1)  # Set gpus=0 if no GPU
# Individual scores
comet_scores = model_output.scores
# Average score
avg_comet = sum(comet_scores) / len(comet_scores)

print(f"Average COMET score for ANSWER: {avg_comet:.4f}")

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Predicting DataLoader 0: 100%|████

Average COMET score for ANSWER: 0.6170


## Chunked SBERT + Max Matching + Average

In [10]:
import nltk
nltk.download('punkt')
import pandas as pd
from sentence_transformers import SentenceTransformer, util

[nltk_data] Downloading package punkt to
[nltk_data]     /data/amey_2311cs10/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
PyTorch version 2.6.0 available.
JAX version 0.6.1 available.


In [11]:
# Load SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # or 'all-mpnet-base-v2' for better quality

Use pytorch device_name: cuda:0
Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [12]:
# Fallback if nltk not available
def safe_sent_tokenize(text):
    try:
        import nltk
        return nltk.sent_tokenize(text)
    except:
        # Basic sentence split fallback
        return [s.strip() for s in text.split('.') if s.strip()]

# SBERT similarity function with fallback tokenization
def sbert_similarity(text1, text2):
    sents1 = safe_sent_tokenize(text1)
    sents2 = safe_sent_tokenize(text2)

    emb1 = model.encode(sents1, convert_to_tensor=True)
    emb2 = model.encode(sents2, convert_to_tensor=True)

    sim_matrix = util.pytorch_cos_sim(emb1, emb2)
    ref_to_pred = sim_matrix.max(dim=1).values.mean().item()
    pred_to_ref = sim_matrix.max(dim=0).values.mean().item()

    return (ref_to_pred + pred_to_ref) / 2

#### Answer part

In [ ]:
similarities = []
for idx, row in res.iterrows():
    try:
        score = sbert_similarity(row['Answer'], row['predicted_answer'])
    except Exception as e:
        print(f"Error at row {idx}: {e}")
        score = None
    similarities.append(score)

# Add to DataFrame
df['sbert_similarity'] = similarities

# Compute and print mean similarity (excluding None)
valid_scores = [s for s in similarities if s is not None]
mean_score = sum(valid_scores) / len(valid_scores)
print(f"Mean SBERT Semantic Similarity for ANSWER: {mean_score:.4f}")

#### Reason part

In [ ]:
similarities = []
for idx, row in res.iterrows():
    try:
        score = sbert_similarity(row['Reasoning'], row['predicted_reasoning'])
    except Exception as e:
        print(f"Error at row {idx}: {e}")
        score = None
    similarities.append(score)

# Add to DataFrame
df['sbert_similarity'] = similarities

# Compute and print mean similarity (excluding None)
valid_scores = [s for s in similarities if s is not None]
mean_score = sum(valid_scores) / len(valid_scores)
print(f"Mean SBERT Semantic Similarity REASONING: {mean_score:.4f}")